# Period Finding via QFT

Demonstrates how Quantum Fourier Transform (QFT) is used for **phase estimation** and how it connects to **period finding**, which is the core quantum component inside Shor’s factoring algorithm.

Steps: 
1. Build a pure NumPy Quantum Phase Estimation (QPE) simulator
2. Observe how QFT extracts periodic information
3. Relate the extracted period to integer factoring
4. Connect the idea to RSA cryptography

---

## Why ?

Modern RSA encryption depends on the difficulty of factoring large numbers.

Classical:
- Factoring large integers is extremely slow.

Quantum:
- Shor’s Algorithm reduces factoring to **period finding**.
- QFT efficiently extracts that period.

In [1]:
# IMPORTS
import math
import numpy as np
from collections import Counter

# Idea

Quantum Phase Estimation (QPE) estimates a hidden phase:

\[
U|u\rangle = e^{2\pi i \phi}|u\rangle
\]

Where:
- \(U\) is a unitary operator
- \(|u\rangle\) is an eigenvector
- \(\phi\) is the hidden phase

The Quantum Fourier Transform (QFT) converts phase information
into measurable probabilities.

In Shor’s Algorithm:

- Periodic modular arithmetic creates hidden phases
- QFT extracts those periods efficiently
- The period reveals factors of composite integers


In [2]:
def qft_matrix(n: int) -> np.ndarray:
    """
    Returns n-qubit QFT unitary matrix.
    Size = 2^n × 2^n
    """
    
    N = 2 ** n
    omega = np.exp(2j * np.pi / N)
    
    F = np.array([
        [omega ** (j * k) for k in range(N)]
        for j in range(N)
    ]) / np.sqrt(N)

    return F

# test qft_matrix 
qft_matrix(2)

array([[ 5.00000000e-01+0.0000000e+00j,  5.00000000e-01+0.0000000e+00j,
         5.00000000e-01+0.0000000e+00j,  5.00000000e-01+0.0000000e+00j],
       [ 5.00000000e-01+0.0000000e+00j,  3.06161700e-17+5.0000000e-01j,
        -5.00000000e-01+6.1232340e-17j, -9.18485099e-17-5.0000000e-01j],
       [ 5.00000000e-01+0.0000000e+00j, -5.00000000e-01+6.1232340e-17j,
         5.00000000e-01-1.2246468e-16j, -5.00000000e-01+1.8369702e-16j],
       [ 5.00000000e-01+0.0000000e+00j, -9.18485099e-17-5.0000000e-01j,
        -5.00000000e-01+1.8369702e-16j,  2.75545530e-16+5.0000000e-01j]])

In [3]:
import plotly.express as px

F = qft_matrix(3)

fig = px.imshow(
    np.abs(F),
    text_auto=".2f",
    title="Magnitude of 3-Qubit QFT Matrix"
)

fig.show()

# Quantum Phase Estimation (QPE)

QPE estimates a phase \(\phi\) using:

1. Superposition
2. Controlled powers of a unitary operator
3. Inverse QFT
4. Measurement

The result concentrates probability around:

\[
m \approx \phi \cdot 2^n
\]

where:
- \(n\) = number of counting qubits
- \(m\) = measured integer

The output amplitudes follow a Dirichlet kernel distribution:

The probability of measuring state \(m\) is:

\[
P(m) = |A(m)|^2
\]

The peak reveals the hidden phase.

# Implementing the QPE Simulator

This simulator:
- Creates the expected QPE probability distribution
- Samples measurement outcomes
- Estimates the hidden phase

In [4]:
def simulate_qpe(theta: float, n_count: int = 4, shots: int = 4096):
    """
    Exact statevector QPE simulation.
    """

    N = 2 ** n_count

    # Convert rotation angle into phase
    true_phase = (theta / (2 * math.pi)) % 1.0

    phi_int = true_phase * N
    m_vals = np.arange(N)

    def amplitude(m):

        delta = phi_int - m
        # Exact representable phase
        if abs(delta % N) < 1e-9:
            return 1.0 + 0j

        return (
            (1 / N)
            * (1 - np.exp(2j * np.pi * delta))
            / (1 - np.exp(2j * np.pi * delta / N))
        )

    amps = np.array([amplitude(m) for m in m_vals])
    probs = np.abs(amps) ** 2
    probs /= probs.sum()

    # Simulated measurements
    samples = np.random.choice(N, size=shots, p=probs)
    counts_raw = Counter(int(s) for s in samples)
    best_m = int(probs.argmax())
    est_phase = best_m / N

    return {
        "true_phase": true_phase,
        "estimated_phase": est_phase,
        "error": abs(est_phase - true_phase),
        "probabilities": probs.tolist(),
        "counts": {
            format(k, f'0{n_count}b'): v
            for k, v in counts_raw.items()
        },
        "n_count": n_count,
        "resolution": 1 / N,
    }

# run example
result = simulate_qpe(math.pi / 4, n_count=4)
result

{'true_phase': 0.125,
 'estimated_phase': 0.125,
 'error': 0.0,
 'probabilities': [1.600156309458075e-33,
  1.5392539862925933e-33,
  1.0,
  1.5392539862925933e-33,
  1.600156309458075e-33,
  1.7082298653091976e-33,
  1.8746997283273227e-33,
  2.118502512538964e-33,
  2.470890769823224e-33,
  2.9842138351085873e-33,
  3.749399456654644e-33,
  4.9330881764039924e-33,
  6.86358547173118e-33,
  1.3565269858397736e-31,
  1.6872297554945895e-32,
  4.865106629828524e-32],
 'counts': {'0010': 4096},
 'n_count': 4,
 'resolution': 0.0625}

In [8]:
import plotly.graph_objects as go

result = simulate_qpe(math.pi / 4, n_count=4)
x_vals = list(range(2 ** result["n_count"]))
fig = go.Figure()

fig.add_trace(go.Bar(
    x=x_vals,
    y=result["probabilities"]
))

fig.update_layout(
    title = "QPE Measurement Probability Distribution",
    xaxis_title = "Measured State",
    yaxis_title = "Probability"
)

fig.show()

# Comparing Multiple Angles

Comparing:
- Exact phases
- Approximate phases
- Aliased outputs

In [5]:
def run_comparison_table(angles, n_count=4, shots=4096):

    N = 2 ** n_count
    res = 1 / N

    header = (
        f"{'θ (rad)':>12}  "
        f"{'True φ':>8}  "
        f"{'Est. φ':>8}  "
        f"{'Error':>8}  "
        f"{'Exact?':>7}"
    )

    print(f"\n{'QFT Phase Estimation Simulator':^60}")
    print(f"n_count={n_count}  ... resolution = {res:.4f}")
    print("─" * 60)
    print(header)
    print("─" * 60)

    for theta in angles:

        r = simulate_qpe(theta, n_count, shots)
        exact = "accurate" if r["error"] < res / 2 else "aliased"

        print(
            f"{theta:>12.5f}  "
            f"{r['true_phase']:>8.5f}  "
            f"{r['estimated_phase']:>8.5f}  "
            f"{r['error']:>8.5f}  "
            f"{exact:>9}"
        )

    print("─" * 60)

In [6]:
# test angles
angles = [
    math.pi / 4,
    math.pi / 3,
    math.pi,
    3 * math.pi / 2,
    1.2,
    0.7,
    5 * math.pi / 7,
]
run_comparison_table(angles, n_count=4)


               QFT Phase Estimation Simulator               
n_count=4  ... resolution = 0.0625
────────────────────────────────────────────────────────────
     θ (rad)    True φ    Est. φ     Error   Exact?
────────────────────────────────────────────────────────────
     0.78540   0.12500   0.12500   0.00000   accurate
     1.04720   0.16667   0.18750   0.02083   accurate
     3.14159   0.50000   0.50000   0.00000   accurate
     4.71239   0.75000   0.75000   0.00000   accurate
     1.20000   0.19099   0.18750   0.00349   accurate
     0.70000   0.11141   0.12500   0.01359   accurate
     2.24399   0.35714   0.37500   0.01786   accurate
────────────────────────────────────────────────────────────


In [7]:
# for 2 qubits
run_comparison_table(angles, n_count = 2)


               QFT Phase Estimation Simulator               
n_count=2  ... resolution = 0.2500
────────────────────────────────────────────────────────────
     θ (rad)    True φ    Est. φ     Error   Exact?
────────────────────────────────────────────────────────────
     0.78540   0.12500   0.00000   0.12500    aliased
     1.04720   0.16667   0.25000   0.08333   accurate
     3.14159   0.50000   0.50000   0.00000   accurate
     4.71239   0.75000   0.75000   0.00000   accurate
     1.20000   0.19099   0.25000   0.05901   accurate
     0.70000   0.11141   0.00000   0.11141   accurate
     2.24399   0.35714   0.25000   0.10714   accurate
────────────────────────────────────────────────────────────


## Visaulization for qubits comparison

In [10]:
theta = math.pi / 3

r2 = simulate_qpe(theta, n_count=2)
r4 = simulate_qpe(theta, n_count=4)


fig = go.Figure()

fig.add_trace(go.Bar(
    x=list(range(2 ** 2)),
    y=r2["probabilities"],
    name="2 qubits"
))
fig.add_trace(go.Bar(
    x=list(range(2 ** 4)),
    y=r4["probabilities"],
    name="4 qubits"
))



fig.update_layout(
    title="Resolution Improvement with More Qubits",
    xaxis_title="Measured State",
    yaxis_title="Probability",
    barmode="group"
)

fig.show()

# Observation

Increasing no of counting qubits:

- Improves precision
- Reduces aliasing
- Produces sharper probability peaks

Shor’s Algorithm scales better with larger quantum registers.

# From Phase Estimation to Period Finding

Shor’s Algorithm studies periodic functions like:

\[
f(x)=a^x \mod N
\]

These functions repeat with some hidden period \(r\).

Example:

\[
2^x \mod 15
\]

produces:

| x | 2^x mod 15 |
|---|---|
|0|1|
|1|2|
|2|4|
|3|8|
|4|1|

The sequence repeats every 4 values.

So:

\[
r = 4
\]

# Connection to RSA Cryptography

RSA security depends on difficulty of factoring:

\[
N = p \times q
\]

where:
- \(p\) and \(q\) are large primes.

Classical:
- Factoring large \(N\) is computationally expensive.

Quantum:
- Shor’s Algorithm can find periods efficiently using QFT.
- Those periods reveal factors.

This threatens traditional RSA encryption.